# Problema 3: Experimentos con una red real

Seleccionamos desde el sitio **Network Repository** tres redes ($G_1$, $G_2$ y $G_3$) pertenecientes a diferentes dominios (excluyendo "Generated Graphs").

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from scipy.io import mmread

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
random.seed(42)

## 3(a) Análisis de Dominios Distintos

### Función de métricas con muestreo estocástico para $\langle l \rangle$
Para evitar el cuello de botella computacional ($O(N^2)$) en la ejecución de caminos mínimos en redes grandes, implementamos un muestreo estocástico tomando 500 nodos de forma aleatoria.

In [ ]:
def calcular_metricas(G, nombre):
    """Calcula las metricas fundamentales de la red, usando muestreo para el camino minimo."""
    # Extraer la Componente Gigante
    if len(G) == 0:
        return {}
    
    largest_cc = max(nx.connected_components(G), key=len)
    G_gigante = G.subgraph(largest_cc)
    
    # Metricas globales
    grados = [d for _, d in G.degree()]
    k_avg = np.mean(grados)
    sigma_k = np.std(grados)
    
    # Camino minimo promedio (<l>)
    if len(G_gigante) <= 3000:
        l_avg = nx.average_shortest_path_length(G_gigante)
    else:
        # Seleccionar muestra aleatoria de 500 nodos de G_gigante
        nodos_muestra = random.sample(list(G_gigante.nodes()), 500)
        
        distancias = []
        for nodo in nodos_muestra:
            # distancias desde este origen hacia todos los demas nodos de G_gigante
            paths = nx.single_source_shortest_path_length(G_gigante, nodo)
            distancias.extend(paths.values())
        
        # Promediamos la gran lista de distancias calculadas
        l_avg = np.mean(distancias) if len(distancias) > 0 else 0
        
    return {
        'Red': nombre,
        'Nodos (N)': G.number_of_nodes(),
        'Aristas (L)': G.number_of_edges(),
        '<k>': round(k_avg, 2),
        '\u03c3_k': round(sigma_k, 2),
        '<l>': round(l_avg, 2)
    }

### Lectura de datos y Tabulación de Métricas (Parte A)

In [ ]:
ruta_a = 'data/redes_reales/parte_a'
archivos_a = os.listdir(ruta_a)

redes_a = []
resultados_a = []

for archivo in archivos_a:
    path_completo = os.path.join(ruta_a, archivo)
    print(f"Cargando {archivo}...")
    
    if archivo.endswith('.edges'):
        G = nx.read_edgelist(path_completo, comments='%', nodetype=int)
    elif archivo.endswith('.mtx'):
        matriz = mmread(path_completo)
        G = nx.Graph(matriz)
    else:
        continue
        
    redes_a.append((archivo, G))
    
    # Calculamos metricas
    metricas = calcular_metricas(G, archivo)
    resultados_a.append(metricas)

df_parte_a = pd.DataFrame(resultados_a)
display(df_parte_a)

### Visualización: Curvas de Distribución de Grados (Log-Binning)
De acuerdo con Barabási, es crucial utilizar escala **Log-Log** y **Log-Binning** (histogramas logarítmicos) para evitar la aparición del "plateau" artificial estocástico en la cola de la distribución.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colores = ['#4CAF50', '#2196F3', '#FF9800']

for i, (nombre, G) in enumerate(redes_a):
    grados = [d for _, d in G.degree()]
    grados = np.array(grados)
    grados = grados[grados > 0] # Evitar log(0)
    
    # Log-Binning con np.logspace
    bins = np.logspace(np.log10(grados.min()), np.log10(grados.max()), 25)
    hist, bin_edges = np.histogram(grados, bins=bins, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Solo graficar bins con densidad mayor a 0
    mask = hist > 0
    
    axes[i].scatter(bin_centers[mask], hist[mask], color=colores[i], s=50, alpha=0.8, edgecolors='k')
    axes[i].set_xscale('log')
    axes[i].set_yscale('log')
    axes[i].set_xlabel('Grado $k$')
    axes[i].set_ylabel('$p_k$ (densidad)')
    axes[i].set_title(f'Distribución de {nombre}', fontsize=12)

plt.suptitle("Distribuciones en Escala Log-Log con Logarithmic Binning", fontsize=15, y=1.05)
plt.tight_layout()
plt.show()

### Conclusión 3(a)

**¿Qué observa? ¿Son redes de libre escala?**

- En la **Red Biológica (bio-yeast-protein)** y la **Red Social (socfb-Wisconsin)** observamos contundentemente un declive lineal en la traza log-log, indicando un comportamiento empírico propio de las **redes de libre escala** obedeciendo fuertemente a una Ley de Potencia ($p_k \sim k^{-\gamma}$). En ellas existe la predominancia de los *"hubs"*: nodos inmensamente conectados que interactúan de anclas en el ecosistema o entorno social.
- En cambio, en la **Red de Infraestructura (road-euroroad)** vemos una decaída abrupta y parabólica, que indica un límite perimetral en torno a unos pocos nodos sin un clúster social preferencial. Las redes espaciales o de carreteras **nunca** son libres de escala porque su naturaleza física y geográfica acota estrictamente la máxima acumulación de aristas en una intersección (límites de construcción).

---
## 3(b) Análisis de un Dominio Común (Redes Facebook Universitarias)
Analizaremos los 4 archivos en el directorio de la **Parte B**, que corresponden al mismo dominio universitario.

In [ ]:
ruta_b = 'data/redes_reales/parte_b'
archivos_b = os.listdir(ruta_b)

resultados_b = []

for archivo in archivos_b:
    path_completo = os.path.join(ruta_b, archivo)
    print(f"Cargando y calculando {archivo}...")
    
    if archivo.endswith('.mtx'):
        matriz = mmread(path_completo)
        G = nx.Graph(matriz)
        
        metricas = calcular_metricas(G, archivo)
        resultados_b.append(metricas)

df_parte_b = pd.DataFrame(resultados_b)
display(df_parte_b)

### Conclusión 3(b)

Al observar los hallazgos en redes originadas íntegramente dentro del dominio social de recintos universitarios americanos (American, Amherst, Bowdoin y Bucknell), las propiedades tabuladas demuestran una **notable estabilidad en sus atributos topológicos**: 

1. Sus grados promedio ($\langle k \rangle$) oscilan de manera coherente, indicando que el número mediano de amistades estudiantiles está modelado orgánicamente dentro de una ventana natural, oscilando entre las 80 a 110 amistades interconectadas.
2. Las desviaciones estándar ($\sigma_k$) son considerables en las cuatro pruebas, probando el constante efecto Scale-Free (*ricos más ricos*) a lo largo del dominio. 
3. Independientemente del abultamiento o tamaño $N$ de cada campus social, todas conservan el fenómeno dictatorial de los **Seis Grados de Separación** o *Mundo Pequeño*; es decir, sus caminos mínimos promedios ($\langle l \rangle$) están sumamente comprimidos por las "cortaduras" provistas a través de los estamentos de estudiantes hiper-conectados.